In [10]:
import sys, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from src.forecasting_utils import (
    FEATURE_COLS, TARGET_COL,
    chronological_split, all_metrics,
)

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping

warnings.filterwarnings("ignore")

In [11]:
df = pd.read_parquet("../data/processed/subsample_features.parquet")
df = df.dropna(subset=FEATURE_COLS + [TARGET_COL])

train, val, test = chronological_split(df)
print(f"Train: {train['date'].min().date()} → {train['date'].max().date()}  rows={len(train):,}")
print(f"Val  : {val['date'].min().date()} → {val['date'].max().date()}  rows={len(val):,}")
print(f"Test : {test['date'].min().date()} → {test['date'].max().date()}  rows={len(test):,}")

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

Train: 2011-03-25 → 2015-12-31  rows=874,986
Val  : 2016-01-01 → 2016-02-29  rows=30,120
Test : 2016-03-01 → 2016-04-24  rows=27,610


In [17]:
# ── Target encoding (mean sales by various groupings, train-only) ────────
# Compute means on TRAIN data only — no leakage from val/test
encodings = {
    "item_mean":       train.groupby("item_id")["sales"].mean(),
    "dept_mean":       train.groupby("dept_id")["sales"].mean(),
    "store_mean":      train.groupby("store_id")["sales"].mean(),
    "state_mean":      train.groupby("state_id")["sales"].mean(),
}
multi_encodings = {
    "item_dow_mean":   train.groupby(["item_id", "day_of_week"])["sales"].mean(),
    "store_dow_mean":  train.groupby(["store_id", "day_of_week"])["sales"].mean(),
    "cat_month_mean":  train.groupby(["cat_id", "month"])["sales"].mean(),
    "item_month_mean": train.groupby(["item_id", "month"])["sales"].mean(),
}

def apply_encodings(d):
    d = d.copy()
    for name, series in encodings.items():
        d[name] = d[series.index.name].map(series)
    for name, series in multi_encodings.items():
        keys = list(series.index.names)
        d[name] = d.set_index(keys).index.map(series)
    return d

train = apply_encodings(train)
val   = apply_encodings(val)
test  = apply_encodings(test)

# Fill any NaN (unseen combinations in val/test) with the global mean
global_mean = train["sales"].mean()
new_cols = list(encodings.keys()) + list(multi_encodings.keys())
for d in [train, val, test]:
    d[new_cols] = d[new_cols].fillna(global_mean)

# Extend FEATURE_COLS so the models pick them up
FEATURE_COLS_EXT = FEATURE_COLS + new_cols
X_train, y_train = train[FEATURE_COLS_EXT], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS_EXT],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS_EXT],  test[TARGET_COL]
print(f"Feature count: {len(FEATURE_COLS_EXT)}  (added {len(new_cols)} target-encoded features)")

Feature count: 33  (added 8 target-encoded features)


In [18]:
lgb_model = LGBMRegressor(
    objective="tweedie",          # ← key change for zero-heavy retail data
    tweedie_variance_power=1.1,   # 1.0=Poisson, 2.0=Gamma; 1.1 is M5-standard
    n_estimators=3000, learning_rate=0.03, num_leaves=127, max_depth=-1,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
    n_jobs=-1, random_state=42,
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(100)],
)
lgb_pred_val  = lgb_model.predict(X_val)
lgb_pred_test = lgb_model.predict(X_test)
joblib.dump(lgb_model, "../models/lightgbm.pkl")
print(all_metrics(y_test, lgb_pred_test, "LightGBM-Tweedie"))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013014 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2709
[LightGBM] [Info] Number of data points in the train set: 874986, number of used features: 33
[LightGBM] [Info] Start training from score 0.018114
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[87]	valid_0's tweedie: 12.0958
{'model': 'LightGBM-Tweedie', 'MAE': 0.9763501352987942, 'RMSE': 1.952907462098132, 'MAPE_pct': 53.66377383389298, 'WMAPE_pct': 75.43803446465469, 'Pred10_pct': 9.218763399365407}


In [19]:
xgb_model = XGBRegressor(
    objective="reg:tweedie", tweedie_variance_power=1.1,
    n_estimators=3000, learning_rate=0.03, max_depth=8,
    subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0,
    tree_method="hist", early_stopping_rounds=100, n_jobs=-1, random_state=42,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_pred_val  = xgb_model.predict(X_val)
xgb_pred_test = xgb_model.predict(X_test)
joblib.dump(xgb_model, "../models/xgboost.pkl")
print(all_metrics(y_test, xgb_pred_test, "XGBoost-Tweedie"))

{'model': 'XGBoost-Tweedie', 'MAE': 0.9703399889821464, 'RMSE': 1.931624592407325, 'MAPE_pct': 53.15008725077237, 'WMAPE_pct': 74.97365840879013, 'Pred10_pct': 9.355972901123403}


In [20]:
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=20, min_samples_leaf=10,
    n_jobs=-1, random_state=42,
)
rf_model.fit(X_train, y_train)
rf_pred_val  = rf_model.predict(X_val)
rf_pred_test = rf_model.predict(X_test)
joblib.dump(rf_model, "../models/random_forest.pkl")
print(all_metrics(y_test, rf_pred_test, "RandomForest"))

{'model': 'RandomForest', 'MAE': 0.9770261096682795, 'RMSE': 1.8791218601497368, 'MAPE_pct': 55.59725006702887, 'WMAPE_pct': 75.4902638605843, 'Pred10_pct': 9.87050853271589}


In [21]:
# Validation predictions (used by the Ridge meta-learner in Task #8)
val_preds = val[["id", "date", "sales"]].copy()
val_preds["pred_lgbm"] = lgb_pred_val
val_preds["pred_xgb"]  = xgb_pred_val
val_preds["pred_rf"]   = rf_pred_val
val_preds.to_parquet("../artifacts/ml_val_predictions.parquet", index=False)

# Test predictions (used by ensemble + OUTL + evaluation)
test_preds = test[["id", "date", "store_id", "state_id", "cat_id", "dept_id", "sales"]].copy()
test_preds["pred_lgbm"] = lgb_pred_test
test_preds["pred_xgb"]  = xgb_pred_test
test_preds["pred_rf"]   = rf_pred_test
test_preds.to_parquet("../artifacts/ml_test_predictions.parquet", index=False)

print("Saved:", val_preds.shape, test_preds.shape)

Saved: (30120, 6) (27610, 10)


In [24]:
results = pd.DataFrame([
    all_metrics(y_test, lgb_pred_test, "LightGBM"),
    all_metrics(y_test, xgb_pred_test, "XGBoost"),
    all_metrics(y_test, rf_pred_test,  "RandomForest"),
])
results.to_csv("../data/processed/ml_model_results.csv", index=False)
display(results)

# Feature importance — use the EXTENDED feature list
fi = pd.DataFrame({
    "feature": FEATURE_COLS_EXT,
    "importance_lgbm": lgb_model.feature_importances_,
    "importance_xgb":  xgb_model.feature_importances_,
    "importance_rf":   rf_model.feature_importances_,
}).sort_values("importance_lgbm", ascending=False)
fi.to_csv("../data/processed/feature_importance.csv", index=False)
display(fi.head(20))

# Save the encoded splits so stacking + quantile notebooks reuse identical features
import json
train.to_parquet("../artifacts/train_encoded.parquet", index=False)
val.to_parquet("../artifacts/val_encoded.parquet",   index=False)
test.to_parquet("../artifacts/test_encoded.parquet", index=False)
with open("../artifacts/feature_cols_ext.json", "w") as f:
    json.dump(FEATURE_COLS_EXT, f)
print("\nSaved encoded splits + feature list to artifacts/")

,model,MAE,RMSE,MAPE_pct,WMAPE_pct,Pred10_pct
0,LightGBM,0.976350,1.952907,53.663774,75.438034,9.218763
1,XGBoost,0.970340,1.931625,53.150087,74.973658,9.355973
2,RandomForest,0.977026,1.879122,55.597250,75.490264,9.870509


,feature,importance_lgbm,importance_xgb,importance_rf
32,item_month_mean,1308,0.024942,0.018768
4,rolling_mean_28,1119,0.134445,0.041827
18,sell_price,1026,0.003911,0.010057
25,item_mean,867,0.005866,0.097038
29,item_dow_mean,716,0.026379,0.136223
3,rolling_mean_7,599,0.620306,0.540287
7,week_of_year,531,0.002604,0.009080
1,lag_14,468,0.006006,0.013709
6,day_of_week,442,0.004411,0.005944
9,year,430,0.006726,0.004713



Saved encoded splits + feature list to artifacts/
